# AI Job Agent — PHASE 1: Data Preparation

**Goal:** Build a clean, structured job corpus that is **completely independent of any user or query**.

This file knows nothing about the user's resume or preferences. Its outputs are consumed later by:

| Output                               | Used in                                  |
| ------------------------------------ | ---------------------------------------- |
| `jobs_prepared.parquet`              | Phase 3 / 4 / 5 (Filtering and Matching) |
| `job_embeddings.npy` + `job_ids.npy` | Phase 3 (Embedding Retrieval)            |
| `skill_vocabulary.parquet`           | Phase 5 (Skill Matching)                 |
| `DATA_DICTIONARY.md`                 | Schema Documentation                     |

**Pipeline:**

```text
Multi-anchor fetch → Normalize text → Language / Spam filters
      → Exact + Near-duplicate removal → Location & date normalization
      → Quality gate → LLM structured extraction (async + cache)
      → Skill normalization → QA → jobs_prepared.parquet
```


---
## 1. Setup

In [ ]:
# Clone the Open Jobs repository
!git clone -q https://github.com/elliottdehn/open-jobs.git 2>/dev/null || echo "already cloned"
%cd /content/open-jobs

!pip install -q uv openai pydantic tqdm nest_asyncio

/content/open-jobs
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 33.5 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import html
import time
import math
import base64
import hashlib
import asyncio
import unicodedata
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 120)

WORK = Path("work")
CACHE = Path("cache")
OUT = Path("outputs")

for d in (WORK, CACHE, OUT):
    d.mkdir(parents=True, exist_ok=True)

print("Ready.")

Ready.


## 2. Configuration

All pipeline decisions are centralized here. There are no scattered values hardcoded throughout the cells.

In [ ]:
CONFIG = {


    "anchors": [
        {
            "name": "data_tech",
            "title": "Data Analyst",
            "location": "Saudi Arabia",
            "text": "Data analysis, business intelligence, SQL, Python, "
                    "dashboards, reporting, data engineering, machine learning, "
                    "software development, backend, frontend, APIs, cloud.",
        },
        {
            "name": "business_ops",
            "title": "Business Analyst",
            "location": "Saudi Arabia",
            "text": "Business analysis, project management, operations, "
                    "strategy, consulting, finance, accounting, human resources, "
                    "recruitment, administration.",
        },
        {
            "name": "engineering_other",
            "title": "Engineer",
            "location": "Saudi Arabia",
            "text": "Mechanical, electrical, civil, industrial engineering, "
                    "maintenance, construction, QA/QC, nursing, medical, "
                    "sales, marketing, customer service.",
        },
    ],


    "fetch_top_groups": 8,

    # ---------- Cleaning thresholds ----------
    "min_jd_chars": 250,
    "max_jd_chars": 25_000,
    "min_title_chars": 3,
    "near_dup_cosine": 0.985,


    "allowed_languages": ["en"],


    "max_age_days": None,

    # ---------- LLM extraction ----------
    "extraction_model": "gpt-5.6-luna",
    "prompt_version": "v2.0",


    "max_jobs_to_extract": None,

    "max_jd_chars_sent_to_llm": 6_000,
    "concurrency": 8,
    "max_retries": 5,
    "checkpoint_every": 200,
}

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
print("RUN_ID:", RUN_ID)

RUN_ID: 20260915_1303


---

## 3. Build the Raw Corpus (Multi-Anchor)

The `open-jobs` tool retrieves groups of jobs that are similar to a reference description. Using multiple anchors provides broader coverage than relying on a single anchor, resulting in a corpus that can serve any future user.

> The `sim` column is produced by the anchor and is not an intrinsic property of the job. It will be removed after merging.


In [ ]:
def fetch_anchor(anchor: dict, top_groups: int) -> pd.DataFrame:
    """Run one open-jobs embed+fetch cycle and return the raw dataframe."""

    anchor_path = WORK / f"anchor_{anchor['name']}.md"
    anchor_path.write_text(
        f"Title: {anchor['title']}\n\n"
        f"Location: {anchor['location']}\n\n"
        f"{anchor['text']}\n",
        encoding="utf-8",
    )

    embed_cmd = (
        f'uv run tools/jobs.py embed '
        f'--file "{anchor_path}" '
        f'--title "{anchor["title"]}" '
        f'--location "{anchor["location"]}"'
    )
    fetch_cmd = f"uv run tools/jobs.py fetch --top {top_groups}"

    rc_embed = os.system(embed_cmd)
    rc_fetch = os.system(fetch_cmd)

    if rc_embed != 0 or rc_fetch != 0:
        print(f"  [WARN] anchor '{anchor['name']}' failed "
              f"(embed={rc_embed}, fetch={rc_fetch})")
        return pd.DataFrame()

    part = pd.read_parquet(WORK / "jobs.parquet")
    part["anchor"] = anchor["name"]


    part.to_parquet(WORK / f"raw_{anchor['name']}.parquet", index=False)
    return part

In [ ]:
frames = []

for anchor in CONFIG["anchors"]:
    cached = WORK / f"raw_{anchor['name']}.parquet"

    if cached.exists():
        part = pd.read_parquet(cached)
        print(f"{anchor['name']:<18} cached   {len(part):>7,} jobs")
    else:
        part = fetch_anchor(anchor, CONFIG["fetch_top_groups"])
        print(f"{anchor['name']:<18} fetched  {len(part):>7,} jobs")

    if len(part):
        frames.append(part)

raw_df = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows before merge-dedup: {len(raw_df):,}")

data_tech          fetched    8,258 jobs
business_ops       fetched    9,292 jobs
engineering_other  fetched   13,398 jobs

Total rows before merge-dedup: 30,948


In [ ]:

raw_df = (
    raw_df
    .sort_values("sim", ascending=False)
    .drop_duplicates(subset=["ats", "slug", "id"], keep="first")
    .reset_index(drop=True)
)


raw_df = raw_df.drop(columns=["sim"], errors="ignore")

print(f"Unique jobs: {len(raw_df):,}")
print(f"Columns: {raw_df.columns.tolist()}")
raw_df.head(3)

Unique jobs: 13,398
Columns: ['ats', 'slug', 'id', 'title', 'company', 'location', 'url', 'seen_ms', 'jd', 'leaf', 'vec_b64', 'pub_ms', 'anchor']


,ats,slug,id,title,company,location,url,seen_ms,jd,leaf,vec_b64,pub_ms,anchor
0,workable,squadio23,924BEE8F59,Business Intelligence Analyst - Saudi National,Squadio,"Riyadh, Riyadh Province, Saudi Arabia",https://apply.workable.com/j/924BEE8F59,1787784706106,Job Description: We are seeking a highly skilled and experienced Business Intelligence Analyst to join our team. Th...,13332,waQbvTVlqjtmBJA9VwQOPJKklTydhli8piQYvYmn9jys5Ji7x4d+ugeFpDs6ZAq9WUSOvLOFOr2GR3Y9rwW6PREkhbw6JSs7+8XDvPQkojw3Zks94cQf...,1.695600e+12,data_tech
1,smartrecruiters,JobsForHumanity,744000060144015,Data Analyst intern,Jobs for Humanity,"Riyadh, Riyadh Province, Saudi Arabia",https://jobs.smartrecruiters.com/JobsForHumanity/744000060144015,1787783320310,"Company Description we looking for Data Analyst, you will support internal teams by transforming raw data into acti...",13257,cSkjvVVJobvMLF09iEtHPPOomjzLijq8p6xau2ZMVj0I7M88qm59PJBJpbxbzWa9cq55vNKM3bz5xwk9uGeFPa5t7Duv60m8TIxUvcLqOT3HZwY97uq8...,1.747493e+12,data_tech
2,smartrecruiters,JobsForHumanity,744000073286960,Business Analyst,Jobs for Humanity,"Riyadh, Riyadh Province, Saudi Arabia",https://jobs.smartrecruiters.com/JobsForHumanity/744000073286960,1787783320310,"Company Description We build full-fledged innovative solutions with a focus on process automation, user experience ...",13332,jDyCvfEbGTxdOXo9FvpeO2Z7Lbshm7e8QfuyvDV6Wj1AG7M8ZtwHPUB5fjy1WW299luYvENaWL0WnBM9ItyRPW/a0bzM2Wm8Rdl9vDW7NDyB3IM9hBsp...,1.753789e+12,engineering_other


---

## 4. Stable Job Identifier

A stable key that does not change between runs. It is essential for deduplication, caching, and linking evaluation results in Phase 9.


In [ ]:
def make_job_id(row) -> str:
    key = f"{row['ats']}|{row['slug']}|{row['id']}"
    return hashlib.sha1(key.encode("utf-8")).hexdigest()[:16]


raw_df["job_id"] = raw_df.apply(make_job_id, axis=1)

assert raw_df["job_id"].is_unique, "job_id collision detected"
print(f"job_id generated for {len(raw_df):,} jobs")
raw_df[["job_id", "title", "company"]].head()

job_id generated for 13,398 jobs


,job_id,title,company
0,b98236511b7615cb,Business Intelligence Analyst - Saudi National,Squadio
1,aee171aaa2d78ca9,Data Analyst intern,Jobs for Humanity
2,651badedd03e277f,Business Analyst,Jobs for Humanity
3,1adc1009dd43f744,Data Analyst/Application Developer (Saudi Arabia),eramtalent-1
4,2411d9cb3784a781,Business Analyst,Jobs for Humanity


---

## 5. Text Normalization

Three versions of each text field, each serving a different purpose:

| Column     | Processing                          | Usage                        |
| ---------- | ----------------------------------- | ---------------------------- |
| `jd_clean` | HTML removed, line breaks preserved | LLM extraction, user display |
| `jd_flat`  | Single line                         | Text similarity, indexing    |
| `jd_raw`   | Original text                       | Auditing and review          |

> **Common mistake:** Applying `\s+ → " "` to everything. This destroys the structure of lists and headings within the job description and reduces extraction quality.


In [ ]:
TAG_RE = re.compile(r"<[^>]{1,400}?>")
SCRIPT_RE = re.compile(r"<(script|style)[^>]*>.*?</\1>", re.S | re.I)
BULLET_RE = re.compile(r"^[\s\u2022\u25cf\u25aa\u00b7\-\*\u2013\u2014]+", re.M)
INLINE_WS_RE = re.compile(r"[ \t\x0b\f\r\u00a0\u2000-\u200a]+")
MANY_NL_RE = re.compile(r"\n{3,}")
ZERO_WIDTH_RE = re.compile(r"[\u200b-\u200f\ufeff\u2060]")


def clean_text(value, keep_newlines: bool = True) -> str:
    """Normalize HTML-ish job text while preserving useful structure."""

    if not isinstance(value, str) or not value.strip():
        return ""

    text = value


    for _ in range(3):
        new = html.unescape(text)
        if new == text:
            break
        text = new

    text = SCRIPT_RE.sub(" ", text)
    text = TAG_RE.sub("\n", text)
    text = ZERO_WIDTH_RE.sub("", text)
    text = unicodedata.normalize("NFKC", text)

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = INLINE_WS_RE.sub(" ", text)

    lines = [ln.strip() for ln in text.split("\n")]
    lines = [ln for ln in lines if ln]
    text = "\n".join(lines)
    text = MANY_NL_RE.sub("\n\n", text)

    if not keep_newlines:
        text = text.replace("\n", " ")
        text = INLINE_WS_RE.sub(" ", text)

    return text.strip()


def norm_key(value) -> str:
    """Aggressive normalization used only for matching and dedup keys."""

    text = clean_text(value, keep_newlines=False).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
df = raw_df.copy()

df["jd_raw"] = df["jd"]

df["title_clean"] = df["title"].map(lambda x: clean_text(x, keep_newlines=False))
df["company_clean"] = df["company"].map(lambda x: clean_text(x, keep_newlines=False))
df["location_raw"] = df["location"].map(lambda x: clean_text(x, keep_newlines=False))
df["url_clean"] = df["url"].fillna("").astype(str).str.strip()

df["jd_clean"] = df["jd"].map(lambda x: clean_text(x, keep_newlines=True))
df["jd_flat"] = df["jd_clean"].map(lambda x: clean_text(x, keep_newlines=False))

df["title_norm"] = df["title_clean"].map(norm_key)
df["company_norm"] = df["company_clean"].map(norm_key)

df["jd_chars"] = df["jd_clean"].str.len()
df["jd_words"] = df["jd_flat"].str.split().map(len)

print(f"Rows: {len(df):,}")
df[["title_clean", "company_clean", "jd_chars", "jd_words"]].describe(include="all").head(4)

Rows: 13,398


,title_clean,company_clean,jd_chars,jd_words
count,13398,13398,13398.0,13398.0
unique,10731,3973,NaN,NaN
top,Data Scientist,kuwait,NaN,NaN
freq,301,514,NaN,NaN


In [ ]:
# تحقّق: لم تعد كيانات HTML موجودة
leftover = df["jd_clean"].str.contains(r"&(amp|lt|gt|nbsp|#\d+);", regex=True, na=False).sum()
tags_left = df["jd_clean"].str.contains(r"<[a-zA-Z/][^>]*>", regex=True, na=False).sum()

print("Rows with leftover HTML entities:", leftover)
print("Rows with leftover HTML tags   :", tags_left)

print("\n--- sample ---")
print(df["jd_clean"].iloc[0][:600])

Rows with leftover HTML entities: 0
Rows with leftover HTML tags   : 0

--- sample ---
Job Description: We are seeking a highly skilled and experienced Business Intelligence Analyst to join our team. The ideal candidate will have a strong analytical mind and excellent communication skills. As a Business Intelligence Analyst, you will be responsible for analyzing and interpreting complex data sets, creating insightful reports and dashboards, and making recommendations to improve business performance. Responsibilities: Analyze complex data sets to identify trends and patterns Develop and maintain reports, dashboards, and visualizations to communicate findings Collaborate with cros


/tmp/ipykernel_719/735209564.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  leftover = df["jd_clean"].str.contains(r"&(amp|lt|gt|nbsp|#\d+);", regex=True, na=False).sum()


---

## 6. Language Detection

The corpus contains Arabic, English, and mixed-language job postings. Without distinguishing between languages, embeddings can become mixed and extraction quality can suffer. A character-ratio approach is sufficient here and does not require an external library.


In [ ]:
AR_CHARS = re.compile(r"[\u0600-\u06FF\u0750-\u077F]")
LAT_CHARS = re.compile(r"[A-Za-z]")


def detect_lang(text: str) -> str:
    if not text:
        return "unknown"

    n_ar = len(AR_CHARS.findall(text))
    n_lat = len(LAT_CHARS.findall(text))
    total = n_ar + n_lat

    if total < 30:
        return "unknown"

    ar_ratio = n_ar / total

    if ar_ratio >= 0.60:
        return "ar"
    if ar_ratio <= 0.10:
        return "en"
    return "mixed"


df["jd_lang"] = df["jd_flat"].map(detect_lang)
print(df["jd_lang"].value_counts())

jd_lang
en         11975
ar          1192
unknown      160
mixed         71
Name: count, dtype: int64


---

## 7. Spam and Aggregator Filtering

The corpus contains aggregated listings from intermediary websites: a single title covering dozens of jobs, or marketing pages with little to no actual job content. These listings heavily contaminate retrieval because they can superficially match almost any query.

We flag them first and filter them afterward, so we can review what was removed.


In [ ]:
SPAM_TITLE_PATTERNS = [
    r"\bmultiple jobs\b",
    r"\bjobs? (in|for) .{0,40}\b20\d\d\b",
    r"\bapply now\b",
    r"\bverified overseas\b",
    r"\bwalk[- ]?in\b",
    r"\burgent(ly)? (hiring|required)\b",
    r"\bjob vacancies?\b",
    r"\bfree recruitment\b",
    r"\bvisa (sponsorship )?available\b",
    r"\bhiring now\b.*\bjobs\b",
    r"\b\d{2,} (job )?(vacancies|openings|positions)\b",
]

SPAM_JD_PATTERNS = [
    r"\bwhatsapp\b.{0,30}\+?\d[\d\s\-]{7,}",
    r"\bsend (your )?cv (to|on) .{0,40}@",
    r"\bregistration fee\b",
    r"\bno (registration )?(fee|charges)\b.{0,40}\bagent\b",
]

SPAM_TITLE_RE = re.compile("|".join(SPAM_TITLE_PATTERNS), re.I)
SPAM_JD_RE = re.compile("|".join(SPAM_JD_PATTERNS), re.I)


def count_title_like_lines(jd: str) -> int:
    """Aggregator pages list many roles as bare short lines."""

    lines = [ln.strip() for ln in jd.split("\n") if ln.strip()]
    short = [
        ln for ln in lines
        if 8 <= len(ln) <= 45 and not ln.endswith((".", ":", ";"))
    ]
    return len(short)


df["spam_title_hit"] = df["title_clean"].str.contains(SPAM_TITLE_RE, na=False)
df["spam_jd_hit"] = df["jd_flat"].str.contains(SPAM_JD_RE, na=False)
df["title_like_lines"] = df["jd_clean"].map(count_title_like_lines)

df["is_spam"] = (
    df["spam_title_hit"]
    | df["spam_jd_hit"]
    | (df["title_like_lines"] >= 25)
)

print(f"Flagged as spam: {df['is_spam'].sum():,} / {len(df):,}")
df.loc[df["is_spam"], ["title_clean", "company_clean", "title_like_lines"]].head(15)

/tmp/ipykernel_719/1797206980.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["spam_title_hit"] = df["title_clean"].str.contains(SPAM_TITLE_RE, na=False)
/tmp/ipykernel_719/1797206980.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["spam_jd_hit"] = df["jd_flat"].str.contains(SPAM_JD_RE, na=False)


Flagged as spam: 89 / 13,398


,title_clean,company_clean,title_like_lines
149,Engineer Jobs in Saudi Arabia 2025,nokryan.com,0
151,Mechanical Engineer Jobs in Saudi Arabia 2025,nokryan.com,0
200,Latest Engineering Jobs in Saudi Arabia 2025,nokryan.com,0
272,Project Manager,Jobs for Humanity,0
319,Latest Engineering jobs in Saudi Arabia 2025 – Apply now,nokryan.com,0
407,Latest Engineering & Technician Jobs in Saudi Arabia 2025,nokryan.com,0
425,Communication Engineer Jobs in K 2025 – Apply Now,nokryan.com,0
480,Junior Site Engineer (Fit-Out) – Urgent Hiring,Jobs for Humanity,0
498,Multiple Jobs in Saudi Arabia 2025 – Apply Now,nokryan.com,0
563,Electronic Engineer Jobs in Saudi Arabia 2025,nokryan.com,0


In [ ]:

sample_spam = df.loc[df["is_spam"]].sample(
    n=min(5, int(df["is_spam"].sum())),
    random_state=42,
) if df["is_spam"].any() else df.head(0)

for _, row in sample_spam.iterrows():
    print("=" * 90)
    print("TITLE  :", row["title_clean"])
    print("COMPANY:", row["company_clean"])
    print("REASON :",
          "title" if row["spam_title_hit"] else "",
          "jd" if row["spam_jd_hit"] else "",
          f"lines={row['title_like_lines']}")
    print(row["jd_clean"][:400])

TITLE  : Visual Merchandiser Jobs in Saudi Arabia
COMPANY: emeraldislemanpower.com
REASON :  jd lines=0
Job Description Visual Merchandisers craft engaging store layouts and dynamic visual displays that captivate retail customers. Additionally, these creative professionals drive brand engagement and boost retail sales across premier shopping locations in Saudi Arabia. Qualified candidates will design floor plans, arrange promotional signage, and style product showcases to maintain international reta
TITLE  : Saudi Arabia Jobs 2025 – Apply Now for Overseas Vacancies
COMPANY: nokryan.com
REASON : title  lines=0
Saudi Arabia Jobs 2025 – Apply Now for Verified Overseas Vacancies If you are searching for top-ranking Saudi Arabia jobs 2025 , this verified overseas job opportunity can help you move toward a stable career. Moreover , the hiring company in KSA is offering multiple roles with safe working conditions and a legal recruitment process. Therefore , you can apply now with full confide

---

## 8. Duplicate Removal

Three layers, ordered from cheapest to most expensive:

1. **Exact URL match**: The same listing was retrieved more than once.

2. **Content fingerprint**: `sha1(title_norm + jd_norm)`

3. **Near-duplicate detection**: The same job posted through different agencies, detected using embedding similarity within groups sharing the same normalized title.

The third layer is restricted to normalized-title groups, avoiding the need to compute a full `n²` similarity matrix.


In [ ]:
before = len(df)


mask_url = df["url_clean"].ne("")
df = pd.concat([
    df[~mask_url],
    df[mask_url].drop_duplicates(subset=["url_clean"], keep="first"),
]).sort_index()

print(f"After URL dedup      : {len(df):,}  (-{before - len(df):,})")


before = len(df)
df["content_hash"] = (
    df["title_norm"] + "||" + df["jd_flat"].map(norm_key)
).map(lambda s: hashlib.sha1(s.encode("utf-8")).hexdigest())

df = df.drop_duplicates(subset=["content_hash"], keep="first")
print(f"After content dedup  : {len(df):,}  (-{before - len(df):,})")

df = df.reset_index(drop=True)

After URL dedup      : 13,300  (-98)
After content dedup  : 12,868  (-432)


In [ ]:
def decode_vec(b64: str, dim: int = 1536) -> np.ndarray:
    """open-jobs stores float32 little-endian vectors as base64."""

    if not isinstance(b64, str) or not b64:
        return np.zeros(dim, dtype=np.float32)

    raw = base64.b64decode(b64)
    vec = np.frombuffer(raw, dtype="<f4")
    return vec.astype(np.float32)


vectors = np.vstack(df["vec_b64"].map(decode_vec).values)

norms = np.linalg.norm(vectors, axis=1, keepdims=True)
norms[norms == 0] = 1.0
vectors_unit = vectors / norms

print("Embedding matrix:", vectors.shape)

Embedding matrix: (12868, 1536)


In [ ]:
def near_duplicate_ids(frame: pd.DataFrame,
                       unit_vectors: np.ndarray,
                       threshold: float) -> set:
    """Find near-duplicate rows within identical normalized titles."""

    drop = set()
    groups = frame.groupby("title_norm").indices

    for _, idx in tqdm(groups.items(), desc="near-dup", total=len(groups)):
        if len(idx) < 2 or len(idx) > 400:
            continue

        sub = unit_vectors[idx]
        sim = sub @ sub.T
        np.fill_diagonal(sim, 0.0)

        kept = []
        for local_i in range(len(idx)):
            if any(sim[local_i, k] >= threshold for k in kept):
                drop.add(int(idx[local_i]))
            else:
                kept.append(local_i)

    return drop


dup_idx = near_duplicate_ids(df, vectors_unit, CONFIG["near_dup_cosine"])
print(f"\nNear-duplicates found: {len(dup_idx):,}")

if dup_idx:
    keep_mask = ~df.index.isin(dup_idx)
    df = df[keep_mask].reset_index(drop=True)
    vectors = vectors[keep_mask]
    vectors_unit = vectors_unit[keep_mask]

print(f"After near-dup removal: {len(df):,}")

near-dup:   0%|          | 0/10602 [00:00<?, ?it/s]


Near-duplicates found: 105
After near-dup removal: 12,763


---

## 9. Location Normalization

Without `country` and `city` as separate columns, strict filtering in Phase 4 is impossible.

Raw location values are inconsistent, for example: `Riyadh, SA`, `Dhahran, sa, Saudi Arabia`, and `Khamis, 7, Saudi Arabia`.


In [ ]:
COUNTRY_ALIASES = {
    "saudi arabia": "Saudi Arabia", "ksa": "Saudi Arabia",
    "sa": "Saudi Arabia", "sau": "Saudi Arabia",
    "kingdom of saudi arabia": "Saudi Arabia",
    "united arab emirates": "UAE", "uae": "UAE", "ae": "UAE",
    "qatar": "Qatar", "qa": "Qatar",
    "kuwait": "Kuwait", "kw": "Kuwait",
    "bahrain": "Bahrain", "bh": "Bahrain",
    "oman": "Oman", "om": "Oman",
    "egypt": "Egypt", "eg": "Egypt",
    "jordan": "Jordan", "jo": "Jordan",
    "united states": "United States", "usa": "United States",
    "us": "United States", "united kingdom": "United Kingdom",
    "uk": "United Kingdom", "india": "India", "in": "India",
    "pakistan": "Pakistan", "germany": "Germany", "france": "France",
    "canada": "Canada", "remote": None,
}

CITY_ALIASES = {
    "riyadh": ("Riyadh", "Riyadh Province"),
    "ar riyadh": ("Riyadh", "Riyadh Province"),
    "al riyadh": ("Riyadh", "Riyadh Province"),
    "jeddah": ("Jeddah", "Makkah Province"),
    "jiddah": ("Jeddah", "Makkah Province"),
    "makkah": ("Makkah", "Makkah Province"),
    "mecca": ("Makkah", "Makkah Province"),
    "madinah": ("Madinah", "Madinah Province"),
    "medina": ("Madinah", "Madinah Province"),
    "dammam": ("Dammam", "Eastern Province"),
    "dhahran": ("Dhahran", "Eastern Province"),
    "khobar": ("Al Khobar", "Eastern Province"),
    "al khobar": ("Al Khobar", "Eastern Province"),
    "jubail": ("Jubail", "Eastern Province"),
    "yanbu": ("Yanbu", "Madinah Province"),
    "tabuk": ("Tabuk", "Tabuk Province"),
    "abha": ("Abha", "Asir Province"),
    "khamis": ("Khamis Mushait", "Asir Province"),
    "khamis mushait": ("Khamis Mushait", "Asir Province"),
    "neom": ("NEOM", "Tabuk Province"),
    "qassim": ("Qassim", "Qassim Province"),
    "buraidah": ("Buraidah", "Qassim Province"),
    "hail": ("Hail", "Hail Province"),
    "jazan": ("Jazan", "Jazan Province"),
    "najran": ("Najran", "Najran Province"),
    "taif": ("Taif", "Makkah Province"),
    "dubai": ("Dubai", "Dubai"),
    "abu dhabi": ("Abu Dhabi", "Abu Dhabi"),
    "sharjah": ("Sharjah", "Sharjah"),
    "doha": ("Doha", "Doha"),
    "kuwait city": ("Kuwait City", "Al Asimah"),
    "manama": ("Manama", "Capital"),
    "muscat": ("Muscat", "Muscat"),
    "cairo": ("Cairo", "Cairo"),
}

REMOTE_RE = re.compile(r"\b(remote|work from home|wfh|anywhere)\b", re.I)
HYBRID_RE = re.compile(r"\bhybrid\b", re.I)


def parse_location(value: str):
    """Return (city, region, country, location_is_remote)."""

    if not value:
        return (None, None, None, False)

    is_remote = bool(REMOTE_RE.search(value))

    parts = [p.strip() for p in re.split(r"[,/|]", value) if p.strip()]
    parts = [p for p in parts if not re.fullmatch(r"\d+", p)]

    city = region = country = None

    for part in parts:
        key = part.lower().strip()

        if country is None and key in COUNTRY_ALIASES:
            mapped = COUNTRY_ALIASES[key]
            if mapped:
                country = mapped
            continue

        if city is None and key in CITY_ALIASES:
            city, region = CITY_ALIASES[key]
            continue


    if country is None and city is not None:
        for key, (c, _) in CITY_ALIASES.items():
            if c == city:
                if key in ("dubai", "abu dhabi", "sharjah"):
                    country = "UAE"
                elif key == "doha":
                    country = "Qatar"
                elif key == "kuwait city":
                    country = "Kuwait"
                elif key == "manama":
                    country = "Bahrain"
                elif key == "muscat":
                    country = "Oman"
                elif key == "cairo":
                    country = "Egypt"
                else:
                    country = "Saudi Arabia"
                break

    if city is None and parts:
        candidate = REMOTE_RE.sub(" ", parts[0])
        candidate = re.sub(r"\s*[-–—]\s*", " ", candidate).strip()
        if candidate.lower() not in COUNTRY_ALIASES and len(candidate) > 2:
            city = candidate.title()

    return (city, region, country, is_remote)


parsed = df["location_raw"].map(parse_location)
df["city"] = parsed.map(lambda t: t[0])
df["region"] = parsed.map(lambda t: t[1])
df["country"] = parsed.map(lambda t: t[2])
df["location_is_remote"] = parsed.map(lambda t: t[3])

print("Country coverage:")
print(df["country"].value_counts(dropna=False).head(10))
print("\nCity coverage (top 12):")
print(df["city"].value_counts(dropna=False).head(12))

Country coverage:
country
Saudi Arabia      4440
None              3121
UAE               1378
United States     1166
Qatar              819
Kuwait             601
Bahrain            301
Oman               203
India              174
United Kingdom     130
Name: count, dtype: int64

City coverage (top 12):
city
None         2537
Riyadh       1748
Dubai         990
Doha          639
Abu Dhabi     264
Manama        258
Jeddah        256
الرياض        250
Ph; Ph        216
Ph; Sa        199
Neom City     180
Al Khobar     179
Name: count, dtype: int64


---
## 10. Dates and freshness

In [ ]:
now_utc = pd.Timestamp.now(tz="UTC")

for src, dst in [("seen_ms", "first_seen_at"), ("pub_ms", "published_at")]:
    if src in df.columns:
        df[dst] = pd.to_datetime(df[src], unit="ms", errors="coerce", utc=True)
    else:
        df[dst] = pd.NaT

df["posted_at"] = df["published_at"].fillna(df["first_seen_at"])
df["age_days"] = (now_utc - df["posted_at"]).dt.total_seconds() / 86400
df["age_days"] = df["age_days"].round(1)

print("Missing posted_at :", int(df["posted_at"].isna().sum()))
print("Oldest posting    :", df["posted_at"].min())
print("Newest posting    :", df["posted_at"].max())
print()
print(df["age_days"].describe().round(1).to_string())

if CONFIG["max_age_days"] is not None:
    before = len(df)
    keep = df["age_days"].isna() | (df["age_days"] <= CONFIG["max_age_days"])
    df, vectors, vectors_unit = df[keep], vectors[keep.values], vectors_unit[keep.values]
    df = df.reset_index(drop=True)
    print(f"Freshness filter: {before:,} -> {len(df):,}")

Missing posted_at : 0
Oldest posting    : 2012-03-07 00:00:00+00:00
Newest posting    : 2026-10-07 00:00:00+00:00

count    12763.0
mean       533.1
std        758.9
min        -21.5
25%         58.8
50%        252.9
75%        757.2
max       5305.5


---

## 11. Quality Gate

All filters are applied in a single batch, with a report documenting the reason each row was excluded.


In [ ]:
rules = {
    "empty_title": df["title_clean"].str.len() < CONFIG["min_title_chars"],
    "empty_jd": df["jd_clean"].str.len() == 0,
    "jd_too_short": df["jd_clean"].str.len() < CONFIG["min_jd_chars"],
    "jd_too_long": df["jd_clean"].str.len() > CONFIG["max_jd_chars"],
    "spam": df["is_spam"],
    "no_url": df["url_clean"].eq(""),
}

if CONFIG["allowed_languages"]:
    rules["language_excluded"] = ~df["jd_lang"].isin(CONFIG["allowed_languages"])

report = pd.DataFrame({
    "rule": list(rules.keys()),
    "rows_hit": [int(m.sum()) for m in rules.values()],
})
report["pct"] = (report["rows_hit"] / len(df) * 100).round(2)

print(f"Rows entering quality gate: {len(df):,}\n")
print(report.sort_values("rows_hit", ascending=False).to_string(index=False))

Rows entering quality gate: 12,763

             rule  rows_hit   pct
     jd_too_short      2468 19.34
language_excluded      1416 11.09
             spam        87  0.68
         empty_jd        14  0.11
      empty_title         1  0.01
      jd_too_long         0  0.00
           no_url         0  0.00


In [ ]:
drop_mask = np.zeros(len(df), dtype=bool)
for mask in rules.values():
    drop_mask |= mask.values

rejected_df = df[drop_mask].copy()
rejected_df.to_parquet(OUT / "rejected_jobs.parquet", index=False)

keep_mask = ~drop_mask
df = df[keep_mask].reset_index(drop=True)
vectors = vectors[keep_mask]
vectors_unit = vectors_unit[keep_mask]

print(f"Rejected : {len(rejected_df):,}  (saved to outputs/rejected_jobs.parquet)")
print(f"Retained : {len(df):,}")
assert len(df) == len(vectors), "dataframe and embedding matrix out of sync"

Rejected : 3,144  (saved to outputs/rejected_jobs.parquet)
Retained : 9,619


In [ ]:

checks = {
    "rows": len(df),
    "unique job_id": df["job_id"].nunique(),
    "duplicate content_hash": int(df["content_hash"].duplicated().sum()),
    "empty titles": int((df["title_clean"] == "").sum()),
    "empty JDs": int((df["jd_clean"] == "").sum()),
    "JDs below threshold": int((df["jd_chars"] < CONFIG["min_jd_chars"]).sum()),
    "missing country": int(df["country"].isna().sum()),
    "missing city": int(df["city"].isna().sum()),
    "spam remaining": int(df["is_spam"].sum()),
}

for key, value in checks.items():
    print(f"{key:<24}: {value:,}")

df[["job_id", "title_clean", "company_clean", "city", "country", "jd_words"]].head(10)

rows                    : 9,619
unique job_id           : 9,619
duplicate content_hash  : 0
empty titles            : 0
empty JDs               : 0
JDs below threshold     : 0
missing country         : 2,381
missing city            : 1,815
spam remaining          : 0


,job_id,title_clean,company_clean,city,country,jd_words
0,b98236511b7615cb,Business Intelligence Analyst - Saudi National,Squadio,Riyadh,Saudi Arabia,162
1,aee171aaa2d78ca9,Data Analyst intern,Jobs for Humanity,Riyadh,Saudi Arabia,68
2,651badedd03e277f,Business Analyst,Jobs for Humanity,Riyadh,Saudi Arabia,430
3,1adc1009dd43f744,Data Analyst/Application Developer (Saudi Arabia),eramtalent-1,Dhahran,Saudi Arabia,143
4,2411d9cb3784a781,Business Analyst,Jobs for Humanity,Riyadh,Saudi Arabia,259
5,fe25b95b2a46ee46,Mechanical Engineer,Jobs for Humanity,Riyadh,Saudi Arabia,120
6,57e2956750463249,Data Analyst,ASSYSTEM,Riyadh,Saudi Arabia,379
7,739c9b68d2787f25,Data Analyst Intern,Jobs for Humanity,Riyadh,Saudi Arabia,129
8,a8bdc6e2f144c147,Data Program Analyst (Saudi Arabia),eramtalent-1,Jeddah,Saudi Arabia,267
9,f94dfbdf0dec5f16,Business Analyst,Confidential,Riyadh,Saudi Arabia,180


---

## 12. Structured Feature Schema (v2)

The key changes compared with the first version:

| Previous Issue                                                                 | Solution                                                                     |
| ------------------------------------------------------------------------------ | ---------------------------------------------------------------------------- |
| `required_skills` returned full sentences (e.g., `"Bachelor's degree in..."`)  | Separate `qualifications` from skills and enforce short phrases of ≤ 4 words |
| A single skill list mixed technical, behavioral, and language skills           | `hard_skills` / `soft_skills` / `tools_technologies` / `languages`           |
| `work_arrangement` was `Not Specified` in 100% of the sample                   | Allow inference and use a separate `*_inferred` flag                         |
| Inconsistency: years of experience = 5 while seniority level = `Not Specified` | Add an explicit consistency rule to the instructions                         |
| No salary or language/nationality requirements                                 | Add dedicated fields                                                         |

The `_inferred` flag is the key: it allows Phase 4 to apply strict filtering to confirmed values, while Phase 5 can assign lower weight to inferred values.


In [ ]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field

ExperienceLevel = Literal[
    "Internship", "Entry Level", "Mid Level", "Senior Level",
    "Executive", "Not Specified",
]

EmploymentType = Literal[
    "Full-time", "Part-time", "Contract", "Internship",
    "Temporary", "Not Specified",
]

WorkArrangement = Literal["On-site", "Remote", "Hybrid", "Not Specified"]

EducationLevel = Literal[
    "High School", "Diploma", "Bachelor's Degree",
    "Master's Degree", "PhD", "Not Specified",
]

SalaryPeriod = Literal["hour", "day", "month", "year", "Not Specified"]


class JobFeatures(BaseModel):
    """Standardized structured representation of a job posting."""

    # ---------- Skills ----------
    hard_skills: List[str] = Field(
        description="Technical or domain skills. Short noun phrases, max 4 words."
    )
    soft_skills: List[str] = Field(
        description="Behavioural or interpersonal skills. Max 4 words each."
    )
    tools_technologies: List[str] = Field(
        description="Named tools, software, platforms, frameworks, standards."
    )
    languages: List[str] = Field(
        description="Spoken/written languages required, e.g. Arabic, English."
    )

    required_skills: List[str] = Field(
        description="Subset of the above that is explicitly mandatory."
    )
    preferred_skills: List[str] = Field(
        description="Subset that is optional, preferred or nice-to-have."
    )

    # ---------- Qualifications (NOT skills) ----------
    qualifications: List[str] = Field(
        description="Certifications, licences, degrees, memberships. Full phrases allowed."
    )

    # ---------- Experience ----------
    experience_level: ExperienceLevel
    experience_level_inferred: bool = Field(
        description="True if derived from wording rather than stated explicitly."
    )
    years_experience_min: Optional[float]
    years_experience_max: Optional[float]

    # ---------- Education ----------
    education_level: List[EducationLevel]
    education_field: List[str]

    # ---------- Employment ----------
    employment_type: EmploymentType
    employment_type_inferred: bool
    work_arrangement: WorkArrangement
    work_arrangement_inferred: bool

    # ---------- Compensation ----------
    salary_min: Optional[float]
    salary_max: Optional[float]
    salary_currency: Optional[str]
    salary_period: SalaryPeriod

    # ---------- Context ----------
    responsibilities: List[str]
    industry: Optional[str]
    department: Optional[str]
    nationality_requirement: Optional[str] = Field(
        description="e.g. 'Saudi nationals only'. Null if none stated."
    )

    # ---------- Self-assessment ----------
    extraction_confidence: float = Field(
        description="0.0-1.0 confidence that the description was informative enough."
    )


print("Schema fields:", len(JobFeatures.model_fields))

Schema fields: 26


### 12.1 Extraction instructions

In [ ]:
SYSTEM_PROMPT = """
You are a job-posting information extraction system. You convert unstructured
job descriptions into a strict structured record.

GENERAL RULES

1. Never invent information. If it is not stated or clearly implied, use an
   empty list, null, or "Not Specified".
2. Work across ALL job domains — nursing, welding, teaching, accounting,
   logistics — not only technology.
3. Extract from the job description only. Ignore company boilerplate,
   EEO statements, and application instructions.

SKILLS — THE MOST IMPORTANT PART

4. A skill is a short noun phrase of AT MOST 4 words.
   GOOD: "financial reporting", "SQL", "patient care", "welding"
   BAD:  "Bachelor's degree in Administration or a relevant field"
   BAD:  "1 to 3 years of experience in similar roles"
   BAD:  "Ability to work independently under pressure in a team"
5. Degrees, years of experience, certifications and licences are NOT skills.
   They belong in `qualifications`, `education_level` or `years_experience_*`.
6. Split skills into four buckets:
   - hard_skills: technical/domain capabilities ("data modeling", "tax accounting")
   - soft_skills: behavioural ("teamwork", "negotiation")
   - tools_technologies: named products/standards ("Power BI", "SAP", "ISO 9001")
   - languages: spoken languages only ("Arabic", "English")
   A skill appears in exactly ONE bucket.
7. Use the canonical name of a tool: "Power BI" not "MS PowerBI" or "powerbi".
8. required_skills and preferred_skills must contain strings that already
   appear in one of the four buckets above, spelled identically.
   If the posting makes no distinction, put everything in required_skills
   and leave preferred_skills empty.

EXPERIENCE

9. years_experience_min / max come from explicit numbers only.
   "3+ years"     -> min 3, max null
   "3 to 5 years" -> min 3, max 5
   "at least 2"   -> min 2, max null
   No number      -> both null. Never assume 0 from "Entry Level".
10. Normalize: Junior/Graduate/Fresh Graduate -> Entry Level;
    Intermediate -> Mid Level; Senior/Lead/Principal/Staff -> Senior Level;
    Director/VP/Head/Chief -> Executive.
11. Consistency: if years_experience_min is known but no level is stated,
    INFER the level (0-1 Entry, 2-4 Mid, 5-9 Senior, 10+ Senior or Executive)
    and set experience_level_inferred = true.
    If the level is stated explicitly, set the flag to false.
    Never return "Not Specified" while a minimum number of years is present.

EMPLOYMENT TYPE AND WORK ARRANGEMENT

12. Return the category, never the raw phrase:
    "2-year contract" -> "Contract"; "full time position" -> "Full-time".
13. If the type is not stated but the posting clearly describes a permanent
    staff role, return "Full-time" with employment_type_inferred = true.
14. work_arrangement:
    - Explicit "remote"/"work from home" -> Remote, inferred = false
    - Explicit "hybrid" -> Hybrid, inferred = false
    - Explicit "on-site"/"in office" -> On-site, inferred = false
    - No statement, but a specific physical workplace, shift, site or
      facility is described -> On-site with work_arrangement_inferred = true
    - Genuinely unclear -> "Not Specified"
    Do NOT return "Not Specified" merely because the exact word is absent.

COMPENSATION

15. Extract salary numbers only if stated. Convert "10k" to 10000.
    salary_currency as an ISO-like code when identifiable: SAR, AED, USD, EUR.
    A range "8,000 - 12,000 SAR/month" -> min 8000, max 12000,
    currency SAR, period "month".

OTHER

16. education_level uses ONLY the allowed enum values. The field of study
    goes in education_field, one entry per field.
17. responsibilities: up to 8 concise bullet phrases describing the work.
18. nationality_requirement: fill only if the posting restricts by nationality
    or residency status.
19. extraction_confidence: 0.9+ for a detailed posting, 0.5 for a thin one,
    below 0.3 if the text is mostly marketing with no real job content.
"""

print(f"Prompt length: {len(SYSTEM_PROMPT):,} chars")

Prompt length: 3,944 chars


---
## 13. Async extraction pipeline

استبدال `.apply()` التسلسلي. الفروق العملية:

- **تزامن** عبر `asyncio` + semaphore — أسرع بعشرات المرات
- **cache دائم** بمفتاح `sha1(jd + model + prompt_version)` — إعادة التشغيل مجانية
- **retry** مع backoff أسّي و jitter، وتسجيل الفشل بدل ابتلاعه
- **checkpoint** كل N وظيفة

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from openai import AsyncOpenAI

try:
    from google.colab import userdata
    api_key = userdata.get("openai_key_new")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")

assert api_key, "OpenAI API key not found"
aclient = AsyncOpenAI(api_key=api_key)

print("Async client ready.")

Async client ready.


In [ ]:
CACHE_PATH = CACHE / f"extraction_{CONFIG['prompt_version']}.jsonl"


def cache_key(jd: str) -> str:
    payload = f"{CONFIG['extraction_model']}|{CONFIG['prompt_version']}|{jd}"
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()


def load_cache() -> dict:
    store = {}
    if CACHE_PATH.exists():
        with CACHE_PATH.open(encoding="utf-8") as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                    store[rec["key"]] = rec["features"]
                except Exception:
                    continue
    return store


EXTRACTION_CACHE = load_cache()
print(f"Cached extractions: {len(EXTRACTION_CACHE):,}")

Cached extractions: 0


In [ ]:
def smart_truncate(jd: str, max_chars: int) -> str:
    """
    Truncate long JDs while preserving both ends.

    Naive head-truncation loses the requirements/skills section, which in
    many postings sits near the bottom (after company boilerplate). This
    keeps ~70% of the budget from the head and ~30% from the tail, joined
    with a marker, so both the role intro and the requirements survive.
    """

    if len(jd) <= max_chars:
        return jd

    head_budget = int(max_chars * 0.7)
    tail_budget = max_chars - head_budget - len("\n[... omitted ...]\n")

    return jd[:head_budget] + "\n[... omitted ...]\n" + jd[-tail_budget:]


_cache_fh = CACHE_PATH.open("a", encoding="utf-8")
_cache_lock = asyncio.Lock()

FAILURES = []


async def extract_one(job_id: str, jd: str, sem: asyncio.Semaphore) -> tuple:
    """Return (job_id, features_dict or None)."""

    jd = smart_truncate(jd, CONFIG["max_jd_chars_sent_to_llm"])
    key = cache_key(jd)

    if key in EXTRACTION_CACHE:
        return job_id, EXTRACTION_CACHE[key]

    delay = 2.0

    async with sem:
        for attempt in range(CONFIG["max_retries"]):
            try:
                response = await aclient.responses.parse(
                    model=CONFIG["extraction_model"],
                    input=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": jd},
                    ],
                    text_format=JobFeatures,
                )

                features = response.output_parsed.model_dump()

                async with _cache_lock:
                    EXTRACTION_CACHE[key] = features
                    _cache_fh.write(
                        json.dumps({"key": key, "job_id": job_id,
                                    "features": features}, ensure_ascii=False) + "\n"
                    )
                    _cache_fh.flush()

                return job_id, features

            except Exception as exc:
                is_last = attempt == CONFIG["max_retries"] - 1

                if is_last:
                    FAILURES.append({"job_id": job_id, "error": repr(exc)[:300]})
                    return job_id, None

                await asyncio.sleep(delay + np.random.rand())
                delay = min(delay * 2, 60)

    return job_id, None

In [ ]:
async def run_extraction(frame: pd.DataFrame) -> dict:
    sem = asyncio.Semaphore(CONFIG["concurrency"])

    tasks = [
        extract_one(row.job_id, row.jd_clean, sem)
        for row in frame.itertuples(index=False)
    ]

    results = {}
    done = 0

    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks),
                     desc="extracting"):
        job_id, features = await coro
        results[job_id] = features
        done += 1

        if done % CONFIG["checkpoint_every"] == 0:
            pd.DataFrame(
                [{"job_id": k, "features": json.dumps(v, ensure_ascii=False)}
                 for k, v in results.items() if v]
            ).to_parquet(WORK / "extraction_checkpoint.parquet", index=False)

    return results

### 13.1 Estimate Tokens and Cost Before Running

Always run this cell before extraction. The fixed number (`FIXED_TOKENS`) represents the size of the `SYSTEM_PROMPT` + JSON schema. It remains the same for every call. This is exactly what most API providers can discount through prompt caching by up to ~90% (`CACHE_DISCOUNT` below), provided that not a single character of the system prompt changes between calls.

Adjust `PRICE_IN_PER_M` and `PRICE_OUT_PER_M` to match the price of the model you are using to estimate the approximate cost in USD before committing to the run.


In [ ]:
FIXED_TOKENS = round((len(SYSTEM_PROMPT) + len(JobFeatures.model_json_schema().__repr__())) / 4)
OUTPUT_TOKENS_PER_JOB = 600
CACHE_DISCOUNT = 0.90

PRICE_IN_PER_M = 0.0
PRICE_OUT_PER_M = 0.0

target_df = df
if CONFIG["max_jobs_to_extract"]:
    target_df = df.head(CONFIG["max_jobs_to_extract"])

already_cached = sum(
    cache_key(smart_truncate(j, CONFIG["max_jd_chars_sent_to_llm"])) in EXTRACTION_CACHE
    for j in target_df["jd_clean"]
)
to_process = len(target_df) - already_cached

avg_jd_tokens = (
    target_df["jd_clean"].str.slice(0, CONFIG["max_jd_chars_sent_to_llm"])
    .str.len().mean() / 4
)

n_truncated = (target_df["jd_clean"].str.len() > CONFIG["max_jd_chars_sent_to_llm"]).sum()
pct_truncated = n_truncated / len(target_df) * 100

input_tokens_no_cache = to_process * (FIXED_TOKENS + avg_jd_tokens)
input_tokens_cached = to_process * (FIXED_TOKENS * (1 - CACHE_DISCOUNT) + avg_jd_tokens)
output_tokens = to_process * OUTPUT_TOKENS_PER_JOB

cost_no_cache = (input_tokens_no_cache / 1e6 * PRICE_IN_PER_M
                 + output_tokens / 1e6 * PRICE_OUT_PER_M)
cost_cached = (input_tokens_cached / 1e6 * PRICE_IN_PER_M
               + output_tokens / 1e6 * PRICE_OUT_PER_M)

print(f"Jobs total          : {len(target_df):,}")
print(f"Already cached (ours): {already_cached:,}")
print(f"To process           : {to_process:,}")
print(f"Fixed tokens/call    : ~{FIXED_TOKENS:,}  (system prompt + schema)")
print(f"Avg JD tokens/call   : ~{avg_jd_tokens:,.0f}")
print(f"Jobs truncated       : {n_truncated:,} ({pct_truncated:.1f}%)  "
      f"[head+tail smart_truncate — requirements section preserved]")
print()
print(f"Input tokens  (no prompt-cache) : {input_tokens_no_cache/1e6:.2f}M")
print(f"Input tokens  (with prompt-cache): {input_tokens_cached/1e6:.2f}M")
print(f"Output tokens                    : {output_tokens/1e6:.2f}M")
print()
if PRICE_IN_PER_M or PRICE_OUT_PER_M:
    print(f"Est. cost (no cache) : ${cost_no_cache:,.2f}")
    print(f"Est. cost (w/ cache) : ${cost_cached:,.2f}")
else:
    print("Set PRICE_IN_PER_M / PRICE_OUT_PER_M above to see cost in $.")

Jobs total          : 9,619
Already cached (ours): 0
To process           : 9,619
Fixed tokens/call    : ~2,021  (system prompt + schema)
Avg JD tokens/call   : ~432
Jobs truncated       : 0 (0.0%)  [head+tail smart_truncate — requirements section preserved]

Input tokens  (no prompt-cache) : 23.60M
Input tokens  (with prompt-cache): 6.10M
Output tokens                    : 5.77M

Set PRICE_IN_PER_M / PRICE_OUT_PER_M above to see cost in $.


> **Before running the next cell:** If `to_process` is large and you do not want to run the full extraction yet, set `CONFIG["max_jobs_to_extract"] = 200` in the configuration cell (Section 2) and restart from there. Start with a small sample first, review the extraction quality in Section 15, then increase the number or remove the limit (`None`) to run the full dataset. Any partial run is saved in the cache and will not be charged again.


In [29]:
target_df = df
if CONFIG["max_jobs_to_extract"]:
    target_df = df.head(CONFIG["max_jobs_to_extract"])

print(f"Jobs to extract : {len(target_df):,}")
print(f"Already cached  : "
      f"{sum(cache_key(smart_truncate(j, CONFIG['max_jd_chars_sent_to_llm'])) in EXTRACTION_CACHE for j in target_df['jd_clean']):,}")

start = time.time()
results = asyncio.get_event_loop().run_until_complete(run_extraction(target_df))
elapsed = time.time() - start

ok = sum(1 for v in results.values() if v)
print(f"\nSucceeded : {ok:,}")
print(f"Failed    : {len(FAILURES):,}")
print(f"Elapsed   : {elapsed/60:.1f} min")

if FAILURES:
    display(pd.DataFrame(FAILURES).head(10))

Jobs to extract : 9,619
Already cached  : 0


extracting:   0%|          | 0/9619 [00:00<?, ?it/s]


Succeeded : 9,619
Failed    : 0
Elapsed   : 168.5 min


### 13.1 Expand features into columns

In [30]:
FEATURE_FIELDS = list(JobFeatures.model_fields.keys())

LIST_FIELDS = [
    f for f in FEATURE_FIELDS
    if str(JobFeatures.model_fields[f].annotation).startswith("typing.List")
]

feat_rows = []
for job_id in df["job_id"]:
    features = results.get(job_id)
    if features is None:
        features = {f: ([] if f in LIST_FIELDS else None) for f in FEATURE_FIELDS}
        features["extraction_status"] = "failed"
    else:
        features = dict(features)
        features["extraction_status"] = "ok"
    features["job_id"] = job_id
    feat_rows.append(features)

feat_df = pd.DataFrame(feat_rows)

prepared_df = df.merge(feat_df, on="job_id", how="left", validate="one_to_one")


for field in LIST_FIELDS:
    prepared_df[field] = prepared_df[field].map(
        lambda v: list(v) if isinstance(v, (list, np.ndarray)) else []
    )

print(f"Rows: {len(prepared_df):,} | Columns: {prepared_df.shape[1]}")
print(prepared_df["extraction_status"].value_counts().to_dict())

Rows: 9,619 | Columns: 66
{'ok': 9619}


---

## 14. Skill Normalization

Without this step, skill matching in Phase 5 fails: `Power BI`, `PowerBI`, and `MS Power BI` would be treated as three different skills when comparing text.


In [31]:
SKILL_ALIASES = {
    # data / bi
    "powerbi": "power bi", "ms power bi": "power bi",
    "microsoft power bi": "power bi",
    "ms excel": "excel", "microsoft excel": "excel",
    "advanced excel": "excel", "ms office": "microsoft office",
    "office suite": "microsoft office", "msoffice": "microsoft office",
    "sql server": "microsoft sql server", "mssql": "microsoft sql server",
    "postgres": "postgresql", "ms sql": "microsoft sql server",
    "google data studio": "looker studio",
    "data visualisation": "data visualization",
    "data analytics": "data analysis",
    "statistical analysis": "statistics",
    "machine learning (ml)": "machine learning",
    "ml": "machine learning", "ai": "artificial intelligence",
    "nlp": "natural language processing",
    "etl pipelines": "etl", "etl processes": "etl",
    # engineering / software
    "js": "javascript", "reactjs": "react", "react.js": "react",
    "nodejs": "node.js", "node js": "node.js",
    "py": "python", "python3": "python",
    "c sharp": "c#", "golang": "go",
    "rest apis": "rest api", "restful api": "rest api",
    "ci cd": "ci/cd", "cicd": "ci/cd",
    "aws cloud": "aws", "amazon web services": "aws",
    "microsoft azure": "azure", "gcp": "google cloud",
    "k8s": "kubernetes",
    # business
    "ms project": "microsoft project",
    "erp systems": "erp", "sap erp": "sap",
    "kpi reporting": "kpi reporting",
    "stakeholder management": "stakeholder management",
    "project mgmt": "project management",
    # soft
    "communication skills": "communication",
    "verbal communication": "communication",
    "written communication": "written communication",
    "team work": "teamwork", "team player": "teamwork",
    "problem-solving": "problem solving",
    "time-management": "time management",
    "attention to detail": "attention to detail",
    "interpersonal skills": "interpersonal skills",
    "leadership skills": "leadership",
    "analytical skills": "analytical thinking",
    "analytical thinking skills": "analytical thinking",
    # languages
    "english language": "english", "arabic language": "arabic",
    "fluent english": "english", "native arabic": "arabic",
    "english (fluent)": "english",
}

ACRONYM_KEEP = {"sql", "aws", "gcp", "erp", "sap", "api", "etl", "bi", "qa",
                "hr", "ui", "ux", "iso", "css", "html", "php", "ios", "crm",
                "kpi", "cad", "plc", "hvac", "ccna", "pmp", "cpa", "acca"}

NOISE_RE = re.compile(
    r"^(ability to|able to|experience (in|with)|knowledge of|proficiency in|"
    r"strong |excellent |good |demonstrated |proven |solid |working )",
    re.I,
)


def normalize_skill(skill: str) -> str:
    """Map a raw skill string to its canonical form, or '' if unusable."""

    if not isinstance(skill, str):
        return ""

    text = clean_text(skill, keep_newlines=False).lower()
    text = NOISE_RE.sub("", text)
    text = re.sub(r"[\(\)\[\]\.,;:]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip(" -–—/")

    if not text or len(text) < 2:
        return ""


    if len(text.split()) > 5:
        return ""

    text = SKILL_ALIASES.get(text, text)

    if text.endswith("s") and text[:-1] in SKILL_ALIASES.values():
        text = text[:-1]

    return SKILL_ALIASES.get(text, text)


def display_skill(canonical: str) -> str:
    if canonical in ACRONYM_KEEP:
        return canonical.upper()
    return " ".join(
        w.upper() if w in ACRONYM_KEEP else w.capitalize()
        for w in canonical.split()
    )


def normalize_skill_list(values) -> list:
    seen, out = set(), []
    for item in (values or []):
        canon = normalize_skill(item)
        if canon and canon not in seen:
            seen.add(canon)
            out.append(canon)
    return out

In [32]:
for field in ["hard_skills", "soft_skills", "tools_technologies",
              "languages", "required_skills", "preferred_skills"]:
    prepared_df[f"{field}_norm"] = prepared_df[field].map(normalize_skill_list)


prepared_df["all_skills_norm"] = prepared_df.apply(
    lambda r: sorted(set(
        r["hard_skills_norm"] + r["tools_technologies_norm"] + r["soft_skills_norm"]
    )),
    axis=1,
)

prepared_df["n_skills"] = prepared_df["all_skills_norm"].map(len)

print("Skills per job:")
print(prepared_df["n_skills"].describe().round(1))

Skills per job:
count    9619.0
mean       15.9
std        10.4
min         0.0
25%         7.0
50%        15.0
75%        22.0
max        81.0
Name: n_skills, dtype: float64


In [33]:
from collections import Counter

counter = Counter()
for skills in prepared_df["all_skills_norm"]:
    counter.update(skills)

vocab_df = (
    pd.DataFrame(counter.most_common(), columns=["skill_canonical", "job_count"])
    .assign(
        display_name=lambda d: d["skill_canonical"].map(display_skill),
        doc_freq=lambda d: (d["job_count"] / len(prepared_df)).round(5),
    )
)


vocab_df["idf"] = np.log(len(prepared_df) / (1 + vocab_df["job_count"])).round(4)

print(f"Vocabulary size: {len(vocab_df):,}")
vocab_df.head(25)

Vocabulary size: 35,890


,skill_canonical,job_count,display_name,doc_freq,idf
0,communication,3336,Communication,0.34681,1.0587
1,problem solving,1858,Problem Solving,0.19316,1.6437
2,collaboration,1338,Collaboration,0.13910,1.9718
3,teamwork,1232,Teamwork,0.12808,2.0543
4,attention to detail,1208,Attention To Detail,0.12558,2.0739
5,analytical thinking,1023,Analytical Thinking,0.10635,2.2400
6,leadership,1010,Leadership,0.10500,2.2528
7,data analysis,911,Data Analysis,0.09471,2.3559
8,machine learning,810,Machine Learning,0.08421,2.4732
9,cross-functional collaboration,766,Cross-functional Collaboration,0.07963,2.5290


---
## 15. Extraction quality assurance

In [34]:
ok_df = prepared_df[prepared_df["extraction_status"] == "ok"]

print(f"Successful extractions: {len(ok_df):,} / {len(prepared_df):,}\n")

coverage = pd.DataFrame([
    {"field": "hard_skills", "filled_pct": (ok_df["hard_skills_norm"].map(len) > 0).mean()},
    {"field": "tools", "filled_pct": (ok_df["tools_technologies_norm"].map(len) > 0).mean()},
    {"field": "experience_level", "filled_pct": (ok_df["experience_level"] != "Not Specified").mean()},
    {"field": "years_experience_min", "filled_pct": ok_df["years_experience_min"].notna().mean()},
    {"field": "education_level", "filled_pct": (ok_df["education_level"].map(len) > 0).mean()},
    {"field": "employment_type", "filled_pct": (ok_df["employment_type"] != "Not Specified").mean()},
    {"field": "work_arrangement", "filled_pct": (ok_df["work_arrangement"] != "Not Specified").mean()},
    {"field": "salary", "filled_pct": ok_df["salary_min"].notna().mean()},
    {"field": "industry", "filled_pct": ok_df["industry"].notna().mean()},
])
coverage["filled_pct"] = (coverage["filled_pct"] * 100).round(1)

print(coverage.to_string(index=False))

Successful extractions: 9,619 / 9,619

               field  filled_pct
         hard_skills        96.9
               tools        56.4
    experience_level        63.3
years_experience_min        49.7
     education_level        53.8
     employment_type        26.2
    work_arrangement        48.1
              salary         5.1
            industry        79.1


In [35]:

for field in ["experience_level", "employment_type", "work_arrangement"]:
    flag = f"{field}_inferred"
    stated = int((~ok_df[flag].astype(bool)).sum())
    inferred = int(ok_df[flag].astype(bool).sum())
    print(f"{field:<20} stated={stated:>6,}  inferred={inferred:>6,}")

print()
for field in ["experience_level", "employment_type", "work_arrangement"]:
    print(f"--- {field} ---")
    print(ok_df[field].value_counts().to_string())
    print()

experience_level     stated= 5,326  inferred= 4,293
employment_type      stated= 8,105  inferred= 1,514
work_arrangement     stated= 5,793  inferred= 3,826

--- experience_level ---
experience_level
Not Specified    3530
Senior Level     3261
Mid Level        1794
Entry Level       596
Executive         376
Internship         62

--- employment_type ---
employment_type
Not Specified    7099
Full-time        2074
Contract          297
Internship        113
Part-time          20
Temporary          16

--- work_arrangement ---
work_arrangement
Not Specified    4989
On-site          4151
Remote            296
Hybrid            183



In [36]:

def grounding_ratio(row) -> float:
    skills = row["hard_skills_norm"] + row["tools_technologies_norm"]
    if not skills:
        return np.nan

    jd_lower = row["jd_flat"].lower()
    hits = sum(1 for s in skills if s.split()[0] in jd_lower)
    return hits / len(skills)


sample = ok_df.sample(n=min(400, len(ok_df)), random_state=42).copy()
sample["grounding"] = sample.apply(grounding_ratio, axis=1)

print("Grounding ratio (extracted skill token present in JD):")
print(sample["grounding"].describe().round(3))

weak = sample[sample["grounding"] < 0.5]
print(f"\nJobs below 0.5 grounding: {len(weak)} / {len(sample)}")
weak[["title_clean", "hard_skills_norm", "grounding"]].head(5)

Grounding ratio (extracted skill token present in JD):
count    388.000
mean       0.952
std        0.110
min        0.000
25%        0.933
50%        1.000
75%        1.000
max        1.000
Name: grounding, dtype: float64

Jobs below 0.5 grounding: 3 / 400


,title_clean,hard_skills_norm,grounding
6414,Lead HR Officer - Human Resource - Qatar | Alshaya,[human resources],0.000000
5993,Offshore electrician,[electrical operations],0.000000
7134,Principal Data Scientist – Rentals Shopping,"[data science, data-driven analysis, opportunity identification]",0.333333


In [37]:

for _, row in ok_df.sample(n=min(3, len(ok_df)), random_state=7).iterrows():
    print("=" * 100)
    print("TITLE      :", row["title_clean"])
    print("COMPANY    :", row["company_clean"])
    print("LOCATION   :", row["city"], "|", row["country"])
    print("CONFIDENCE :", row["extraction_confidence"])
    print("\nJD (first 700 chars):")
    print(row["jd_clean"][:700])
    print("\nEXTRACTED")
    print("  hard_skills :", row["hard_skills_norm"][:12])
    print("  tools       :", row["tools_technologies_norm"][:12])
    print("  soft_skills :", row["soft_skills_norm"][:8])
    print("  languages   :", row["languages_norm"])
    print("  quals       :", row["qualifications"][:4])
    print("  experience  :", row["experience_level"],
          f"(inferred={row['experience_level_inferred']})",
          f"years={row['years_experience_min']}-{row['years_experience_max']}")
    print("  education   :", row["education_level"], row["education_field"])
    print("  employment  :", row["employment_type"], "|", row["work_arrangement"])
    print("  salary      :", row["salary_min"], row["salary_max"],
          row["salary_currency"], row["salary_period"])
    print()

TITLE      : Data Management Assistant BD
COMPANY    : Simera Professional
LOCATION   : Dubai | UAE
CONFIDENCE : 0.91

JD (first 700 chars):
Responsibilities Enter, update, organize, and maintain data across databases, spreadsheets, and internal systems Review information for accuracy, completeness, and consistency Identify and correct data entry errors, duplicates, and inconsistencies Perform regular data validation and quality checks Organize and maintain digital files, records, and documentation Assist with data imports, exports, and system updates Collect and consolidate information from multiple sources Prepare and format spreadsheets, reports, and data summaries Track data-related tasks, updates, and pending information Assist with maintaining accurate customer, employee, operational, or business records Research missing or

EXTRACTED
  hard_skills : ['data management', 'data entry', 'data validation', 'data quality checks', 'records management', 'spreadsheet formatting', 'data c

---
## 16. Save Phase 1 outputs

In [38]:
FINAL_COLUMNS = [
    # identity
    "job_id", "ats", "slug", "id", "url_clean",
    # core text
    "title_clean", "company_clean", "jd_clean", "jd_flat",
    "jd_chars", "jd_words", "jd_lang",
    # location
    "location_raw", "city", "region", "country", "location_is_remote",
    # time
    "first_seen_at", "published_at", "posted_at", "age_days",
    # raw extracted
    "hard_skills", "soft_skills", "tools_technologies", "languages",
    "required_skills", "preferred_skills", "qualifications",
    # normalized
    "hard_skills_norm", "soft_skills_norm", "tools_technologies_norm",
    "languages_norm", "required_skills_norm", "preferred_skills_norm",
    "all_skills_norm", "n_skills",
    # structured
    "experience_level", "experience_level_inferred",
    "years_experience_min", "years_experience_max",
    "education_level", "education_field",
    "employment_type", "employment_type_inferred",
    "work_arrangement", "work_arrangement_inferred",
    "salary_min", "salary_max", "salary_currency", "salary_period",
    "responsibilities", "industry", "department", "nationality_requirement",
    # meta
    "extraction_status", "extraction_confidence", "content_hash",
]

final_df = prepared_df[[c for c in FINAL_COLUMNS if c in prepared_df.columns]].copy()

jobs_path = OUT / "jobs_prepared.parquet"
final_df.to_parquet(jobs_path, index=False, compression="snappy")

print(f"Saved {jobs_path}  ({len(final_df):,} rows, {final_df.shape[1]} cols, "
      f"{jobs_path.stat().st_size/1e6:.1f} MB)")

Saved outputs/jobs_prepared.parquet  (9,619 rows, 57 cols, 24.9 MB)


In [39]:

np.save(OUT / "job_embeddings.npy", vectors_unit.astype(np.float32))
np.save(OUT / "job_ids.npy", final_df["job_id"].to_numpy())

vocab_df.to_parquet(OUT / "skill_vocabulary.parquet", index=False)

meta = {
    "run_id": RUN_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "n_jobs": int(len(final_df)),
    "n_rejected": int(len(rejected_df)),
    "embedding_dim": int(vectors_unit.shape[1]),
    "embedding_model": "text-embedding-3-small",
    "extraction_model": CONFIG["extraction_model"],
    "prompt_version": CONFIG["prompt_version"],
    "extraction_failures": len(FAILURES),
    "vocabulary_size": int(len(vocab_df)),
    "config": CONFIG,
}

(OUT / "run_metadata.json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(json.dumps({k: v for k, v in meta.items() if k != "config"}, indent=2))

{
  "run_id": "20260915_1303",
  "created_utc": "2026-09-15T15:57:36.211082+00:00",
  "n_jobs": 9619,
  "n_rejected": 3144,
  "embedding_dim": 1536,
  "embedding_model": "text-embedding-3-small",
  "extraction_model": "gpt-5.6-luna",
  "prompt_version": "v2.0",
  "extraction_failures": 0,
  "vocabulary_size": 35890
}


In [40]:
DATA_DICTIONARY = """# jobs_prepared.parquet — Data Dictionary

One row per unique job posting. Produced by PHASE 1 and consumed by PHASE 3-5.

## Identity
| Column | Type | Notes |
|---|---|---|
| job_id | str | sha1(ats|slug|id)[:16]. Stable across runs. Join key everywhere. |
| ats / slug / id | str | Source system identifiers. |
| url_clean | str | Canonical application URL. |
| content_hash | str | Fingerprint used for deduplication. |

## Text
| Column | Type | Notes |
|---|---|---|
| title_clean | str | Whitespace-normalized title. |
| company_clean | str | Normalized employer name. |
| jd_clean | str | HTML-decoded description, **line structure preserved**. Use for LLM and display. |
| jd_flat | str | Single-line version. Use for keyword search and similarity. |
| jd_lang | str | en / ar / mixed / unknown. |

## Location
| Column | Type | Notes |
|---|---|---|
| city, region, country | str/null | Parsed. `country` drives hard filtering in PHASE 4. |
| location_is_remote | bool | Remote keyword present in the location string. |

## Time
| Column | Type | Notes |
|---|---|---|
| posted_at | datetime UTC | published_at, falling back to first_seen_at. |
| age_days | float | Days since posted_at at build time. |

## Skills
Raw fields (`hard_skills`, `soft_skills`, `tools_technologies`, `languages`,
`required_skills`, `preferred_skills`) hold the LLM output verbatim.
The `*_norm` variants are canonicalized via the alias map and are the ones
matching should use. `all_skills_norm` is the deduplicated union of
hard skills, tools and soft skills.

## Structured fields
| Column | Notes |
|---|---|
| experience_level | Internship / Entry / Mid / Senior / Executive / Not Specified |
| experience_level_inferred | True = derived, not stated. Weight lower in scoring. |
| years_experience_min/max | Explicit numbers only; null when unstated. |
| education_level | Controlled enum list. |
| employment_type, work_arrangement | Each paired with an `_inferred` flag. |
| salary_min/max/currency/period | Null when not disclosed (the common case). |
| nationality_requirement | e.g. "Saudi nationals only". Null if unrestricted. |
| extraction_confidence | 0-1 self-assessment. Filter below 0.3 for strict use. |

## Companion files
| File | Contents |
|---|---|
| job_embeddings.npy | float32 (n, 1536), L2-normalized, row order == job_ids.npy |
| job_ids.npy | job_id array aligning embeddings to the parquet |
| skill_vocabulary.parquet | canonical skill, job_count, doc_freq, idf |
| rejected_jobs.parquet | Rows removed by the quality gate, with reason columns |
| run_metadata.json | Full config and counts for reproducibility |

## PHASE 4 filtering contract
Hard filters should use: `country`, `city`, `work_arrangement`
(when `work_arrangement_inferred == False`), `years_experience_min`,
`education_level`, `nationality_requirement`, `age_days`.
Never hard-filter on an inferred value.
"""

(OUT / "DATA_DICTIONARY.md").write_text(DATA_DICTIONARY, encoding="utf-8")
print("Data dictionary written.")

Data dictionary written.


In [41]:
# التحقق النهائي: أعد التحميل وتأكد من التوافق
reloaded = pd.read_parquet(OUT / "jobs_prepared.parquet")
emb = np.load(OUT / "job_embeddings.npy")
ids = np.load(OUT / "job_ids.npy", allow_pickle=True)

assert len(reloaded) == len(emb) == len(ids), "row count mismatch"
assert (reloaded["job_id"].to_numpy() == ids).all(), "embedding order mismatch"
assert reloaded["job_id"].is_unique, "duplicate job_id"
assert np.allclose(np.linalg.norm(emb, axis=1), 1.0, atol=1e-3), "vectors not unit-norm"

print("PHASE 1 COMPLETE")
print(f"  jobs            : {len(reloaded):,}")
print(f"  columns         : {reloaded.shape[1]}")
print(f"  embeddings      : {emb.shape}")
print(f"  skill vocabulary: {len(vocab_df):,}")
print(f"  extraction ok   : {(reloaded['extraction_status'] == 'ok').mean():.1%}")
print("\nReady for PHASE 2 (Candidate Profile) and PHASE 3 (Retrieval).")

PHASE 1 COMPLETE
  jobs            : 9,619
  columns         : 57
  embeddings      : (9619, 1536)
  skill vocabulary: 35,890
  extraction ok   : 100.0%

Ready for PHASE 2 (Candidate Profile) and PHASE 3 (Retrieval).


---

## What Changed Compared with the First Version

| #  | Change                                                       | Impact                                                                 |
| -- | ------------------------------------------------------------ | ---------------------------------------------------------------------- |
| 1  | Removed user preferences and `sim` from Phase 1              | The output is now a general-purpose corpus rather than a search result |
| 2  | Multiple anchors instead of a single anchor                  | Broader coverage across domains                                        |
| 3  | Stable `job_id`                                              | Enables deduplication, caching, and linking Phase 9 results            |
| 4  | HTML removal + line-break preservation                       | Eliminates `&amp;` and improves extraction quality                     |
| 5  | Language detection                                           | Separates Arabic and English job postings                              |
| 6  | Spam filtering with a rejection report                       | Removes aggregator listings from retrieval                             |
| 7  | Three-layer deduplication using embeddings                   | Detects the same job posted by different agencies                      |
| 8  | Location normalization to city/region/country                | Makes Phase 4 filtering possible                                       |
| 9  | Separating `qualifications` from skills + four skill buckets | Makes Phase 5 matching meaningful                                      |
| 10 | `_inferred` flags                                            | Enables strict filtering based only on confirmed values                |
| 11 | Async + cache + retry                                        | Reduces runtime from hours to minutes, with free reruns                |
| 12 | Skill vocabulary + IDF                                       | Gives greater weight to rare skills during evaluation                  |
| 13 | Separate embedding storage                                   | Phase 3 does not need to load the text data                            |

## Next Step — PHASE 2

A separate notebook that builds `candidate_profile` using exactly the same schema:

the same skill buckets, the same normalization, and the same enum values. This symmetry makes matching in Phase 5 a straightforward field-by-field comparison rather than requiring additional text processing.
